# Assignment 09: Custom Backward Pass (100 points)

**Unit 06: Programming PyTorch | AI 310**

PyTorch's autograd handles backward passes automatically. But sometimes you need custom gradient computation — for numerical stability, efficiency, or to implement operations that autograd cannot differentiate. This assignment teaches you to use `torch.autograd.Function` to define custom forward and backward passes.

**Notation**:
- `ctx` = context object for saving tensors between forward and backward
- `grad_output` = gradient flowing back from downstream operations

In [ ]:
"""DO NOT MAKE ANY CHANGE IN THIS CELL."""
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

**WARNING**: Do not import any additional libraries.

---

## Part 1 (15 points, coding)

Implement a custom `autograd.Function` for **ReLU**.

Forward: $\text{ReLU}(x) = \max(0, x)$

Backward: $\frac{\partial \text{ReLU}}{\partial x} = \begin{cases} 1 & x > 0 \\ 0 & x \leq 0 \end{cases}$

Use `ctx.save_for_backward()` to save the input tensor.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class CustomReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        pass
    
    @staticmethod
    def backward(ctx, grad_output):
        pass

# Convenience function
custom_relu = CustomReLU.apply

In [ ]:
""" END OF THIS PART """
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = custom_relu(x)
assert torch.allclose(y, torch.tensor([0.0, 0.0, 0.0, 1.0, 2.0]))

y.sum().backward()
assert torch.allclose(x.grad, torch.tensor([0.0, 0.0, 0.0, 1.0, 1.0]))

# Verify with gradcheck
x_check = torch.randn(5, dtype=torch.double, requires_grad=True)
assert torch.autograd.gradcheck(custom_relu, x_check, eps=1e-6)
print("Part 1 passed!")

---

## Part 2 (20 points, coding)

Implement a custom `autograd.Function` for the **stable log-softmax**.

Forward: $\text{LogSoftmax}(z)_i = z_i - \log\sum_j e^{z_j}$

Use the log-sum-exp trick: $\log\sum_j e^{z_j} = m + \log\sum_j e^{z_j - m}$ where $m = \max_j z_j$.

Backward: $\frac{\partial \mathcal{L}}{\partial z_i} = \bar{y}_i - \text{softmax}(z)_i \cdot \sum_j \bar{y}_j$

where $\bar{y} = \frac{\partial \mathcal{L}}{\partial \text{LogSoftmax}(z)}$ is the incoming gradient.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class CustomLogSoftmax(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        """
        input: (B, C) logits
        output: (B, C) log-probabilities
        """
        pass
    
    @staticmethod
    def backward(ctx, grad_output):
        """
        grad_output: (B, C) incoming gradient
        returns: (B, C) gradient w.r.t. input
        """
        pass

custom_log_softmax = CustomLogSoftmax.apply

In [ ]:
""" END OF THIS PART """
x = torch.randn(4, 5, requires_grad=True)
y = custom_log_softmax(x)
y_ref = torch.log_softmax(x, dim=-1)

assert torch.allclose(y, y_ref, atol=1e-5), "Forward pass does not match"

# Test backward
loss = y.sum()
loss.backward()
grad_custom = x.grad.clone()

x2 = x.data.clone().requires_grad_(True)
y2 = torch.log_softmax(x2, dim=-1)
y2.sum().backward()
grad_ref = x2.grad

assert torch.allclose(grad_custom, grad_ref, atol=1e-4), "Backward pass does not match"

# Stability test: large values
x_large = torch.tensor([[1000.0, 999.0, 998.0]], requires_grad=True)
y_large = custom_log_softmax(x_large)
assert not torch.isnan(y_large).any(), "Should be stable for large inputs"

# Gradcheck
x_check = torch.randn(3, 4, dtype=torch.double, requires_grad=True)
assert torch.autograd.gradcheck(custom_log_softmax, x_check, eps=1e-6)
print("Part 2 passed!")

---

## Part 3 (20 points, coding)

Implement a custom `autograd.Function` for the **straight-through estimator** (STE).

The STE is used to backpropagate through discrete operations (e.g., quantization, hard thresholding).

Forward: $y = \text{round}(x)$ (discretize to nearest integer)

Backward: $\frac{\partial y}{\partial x} = 1$ (pretend the rounding did not happen)

This is mathematically incorrect ($\text{round}$ has zero derivative almost everywhere), but it works well in practice for training quantized neural networks.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class StraightThroughEstimator(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        pass
    
    @staticmethod
    def backward(ctx, grad_output):
        pass

ste_round = StraightThroughEstimator.apply

In [ ]:
""" END OF THIS PART """
x = torch.tensor([0.3, 0.7, 1.2, 2.8], requires_grad=True)
y = ste_round(x)

# Forward should round
assert torch.allclose(y, torch.tensor([0.0, 1.0, 1.0, 3.0]))

# Backward should pass through (identity gradient)
y.sum().backward()
assert torch.allclose(x.grad, torch.ones(4)), "STE gradient should be identity"

# Test in a simple optimization: learn x such that round(x) = target
x = torch.tensor([0.1], requires_grad=True)
target = torch.tensor([3.0])
optimizer = torch.optim.SGD([x], lr=0.1)

for _ in range(50):
    optimizer.zero_grad()
    y = ste_round(x)
    loss = (y - target) ** 2
    loss.backward()
    optimizer.step()

assert ste_round(x).item() == 3.0, f"Should converge to round(x)=3, got {ste_round(x).item()}"
print(f"Part 3 passed! x={x.item():.3f}, round(x)={ste_round(x).item()}")

---

## Part 4 (25 points, coding)

Implement a custom `autograd.Function` for **matrix square root** (for a positive definite matrix).

Forward: Given $A$ (positive definite), compute $S = A^{1/2}$ such that $S S = A$.

Use eigendecomposition: $A = Q \Lambda Q^T$, so $A^{1/2} = Q \Lambda^{1/2} Q^T$.

Backward: The gradient of the matrix square root is complex. Use the formula:

$$\frac{\partial \mathcal{L}}{\partial A} = Q \left(F \odot (Q^T \bar{S} Q)\right) Q^T$$

where $\bar{S}$ is the incoming gradient, $\odot$ is element-wise multiplication, and:

$$F_{ij} = \frac{1}{\sqrt{\lambda_i} + \sqrt{\lambda_j}}$$

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class MatrixSquareRoot(torch.autograd.Function):
    @staticmethod
    def forward(ctx, A):
        """
        A: (N, N) positive definite matrix
        Returns: (N, N) matrix square root S where S @ S = A
        """
        pass
    
    @staticmethod
    def backward(ctx, grad_output):
        """
        grad_output: (N, N) incoming gradient dL/dS
        Returns: (N, N) gradient dL/dA
        """
        pass

matrix_sqrt = MatrixSquareRoot.apply

In [ ]:
""" END OF THIS PART """
# Create a positive definite matrix
torch.manual_seed(42)
L = torch.randn(4, 4)
A = L @ L.T + 0.1 * torch.eye(4)   # Positive definite
A = A.requires_grad_(True)

S = matrix_sqrt(A)

# Verify: S @ S should equal A
reconstructed = S @ S
assert torch.allclose(reconstructed, A.detach(), atol=1e-4), \
    f"S @ S should equal A. Max diff: {(reconstructed - A.detach()).abs().max()}"

# Verify backward: use gradcheck
A_check = (L @ L.T + 0.1 * torch.eye(4)).to(torch.double).requires_grad_(True)
assert torch.autograd.gradcheck(matrix_sqrt, A_check, eps=1e-6, atol=1e-3), \
    "Gradient check failed!"

print("Part 4 passed!")

---

## Part 5 (20 points, non-coding)

Answer the following questions:

1. Why does `ctx.save_for_backward()` exist instead of just storing tensors as attributes? What memory optimization does it enable?

2. When would you need a custom `autograd.Function` instead of relying on PyTorch's automatic differentiation?

3. In the Straight-Through Estimator (Part 3), the backward pass does not match the mathematical gradient of the forward pass. Why does this still work for training?

4. What does `torch.autograd.gradcheck` do, and why is it important for verifying custom backward passes?

5. The matrix square root backward (Part 4) uses eigendecomposition. What happens if the matrix has a repeated eigenvalue? How would you handle this numerically?

### WRITE YOUR SOLUTION HERE ###

1. *Your answer here*

2. *Your answer here*

3. *Your answer here*

4. *Your answer here*

5. *Your answer here*

""" END OF THIS PART """